# BioBERT + Focal Loss for Multi-Level Enzyme Classification

This is a curated research notebook of the ISITIA 2025 `Improving BioBERT Performance in Multi-Level Enzyme Classification`.


In [1]:
import pandas as pd
import numpy as np
import torch
from tqdm import tqdm

import math
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.data.dataset import Dataset
from transformers import TrainingArguments, Trainer, AutoTokenizer, AutoModelForSequenceClassification, AutoModel
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

In [2]:
train = pd.read_excel("../data/train_iea.xlsx")
test = pd.read_excel("../data/test_iea.xlsx")

In [3]:
y_train = train[['class', 'subclass', 'subsubclass', 'substrate']]
y_test = test[['class', 'subclass', 'subsubclass', 'substrate']]

In [4]:
train.drop(['class', 'subclass', 'subsubclass', 'substrate'], axis = 1, inplace = True)
test.drop(['class', 'subclass', 'subsubclass', 'substrate'], axis = 1, inplace = True)

In [5]:
train

,enzyme,go_terms,go_name_def
0,Q8WWF5,"'GO:0005789','GO:0005515','GO:0061630','GO:004...","[NAME] endoplasmic reticulum membrane [DEF] ""T..."
1,A5VK99,"'GO:0004417','GO:0046872','GO:0005524','GO:001...",[NAME] hydroxyethylthiazole kinase activity [D...
2,B3L2G0,"'GO:0030488','GO:0008033','GO:0032259','GO:000...","[NAME] tRNA methylation [DEF] ""The posttranscr..."
3,Q12404,"'GO:0005623','GO:0005788','GO:0006457','GO:005...","[NAME] cell [DEF] ""The basic structural and fu..."
4,P20656,"'GO:0030598','GO:0016787','GO:0090729','GO:001...","[NAME] rRNA N-glycosylase activity [DEF] ""Cata..."
...,...,...,...
7702,A1K9L8,"'GO:0005737','GO:0106074','GO:0006418','GO:000...","[NAME] cytoplasm [DEF] ""All of the contents of..."
7703,D7GG24,"'GO:0008080','GO:0016747','GO:0016746','GO:001...","[NAME] N-acetyltransferase activity [DEF] ""Cat..."
7704,P23550,"'GO:0005975','GO:0008152','GO:0030245','GO:000...","[NAME] carbohydrate metabolic process [DEF] ""T..."
7705,P46562,"'GO:0004029','GO:0016491','GO:0016620','GO:004...",[NAME] aldehyde dehydrogenase (NAD+) activity ...


In [152]:
train[train['enzyme'] == 'B3L2G0']['go_name_def'].values

array(['[NAME] tRNA methylation [DEF] "The posttranscriptional addition of methyl groups to specific residues in a tRNA molecule." [GOC:mah] [NAME] tRNA processing [DEF] "The process in which a pre-tRNA molecule is converted to a mature tRNA, ready for addition of an aminoacyl group." [GOC:jl, PMID:12533506] [NAME] methylation [DEF] "The process in which a methyl group is covalently attached to a molecule." [GOC:mah] [NAME] tRNA (guanine-N1-)-methyltransferase activity [DEF] "Catalysis of the reaction: S-adenosyl-L-methionine + tRNA = S-adenosyl-L-homocysteine + tRNA containing N1-methylguanine." [EC:2.1.1.31] [NAME] tRNA (guanine(37)-N(1))-methyltransferase activity [DEF] "Catalysis of the reaction: S-adenosyl-L-methionine + guanine(37) in tRNA = N(1)-methylguanine(37) in tRNA + S-adenosyl-L-homocysteine." [EC:2.1.1.228] [NAME] transferase activity [DEF] "Catalysis of the transfer of a group, e.g. a methyl group, glycosyl group, acyl group, phosphorus-containing, or other groups, from

In [6]:
train['go_name_def'].apply(lambda x: len(x))

0       2836
1       2107
2       2416
3       3207
4       1390
        ... 
7702    2926
7703    1603
7704    1813
7705    2020
7706    5814
Name: go_name_def, Length: 7707, dtype: int64

In [7]:
train_text = train['go_name_def'].tolist()
train_label = y_train.values.tolist()

test_text = test['go_name_def'].tolist()
test_label = y_test.values.tolist()

In [8]:
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-v1.1")

In [9]:
train_encodings = tokenizer(train_text, padding = "max_length", truncation = True, max_length = 512)
test_encodings = tokenizer(test_text, padding = "max_length", truncation = True, max_length = 512)

In [10]:
class TextClassifierDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

In [11]:
train_dataset = TextClassifierDataset(train_encodings, train_label)
test_dataset = TextClassifierDataset(test_encodings, test_label)

In [45]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size=16)

In [13]:
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score

def document_metrics(list_hyp, list_label):
    metrics = {}
    metrics["Accuracy"] = accuracy_score(list_hyp, list_label)
    metrics["F1 Score"] = f1_score(list_hyp, list_label, average = 'macro')
    metrics["Precision"] = precision_score(list_hyp, list_label, average = 'macro')
    metrics["Recall"] = recall_score(list_hyp, list_label, average = 'macro')
    return metrics

def metrics_to_string(metric_dict):
    string_list = []
    for key, value in metric_dict.items():
        string_list.append('{}:{:.3f}'.format(key, value))
    return ' '.join(string_list)

In [ ]:
def focal_loss(batch, onehot, alpha, gamma):
    x = torch.mul(batch, onehot)
    y = torch.sum(x, dim = 1).squeeze()
    lp = torch.abs(torch.log(y))
    eg = torch.pow((1 - y), gamma)
    loss = alpha * (torch.mul(eg, lp))
    return loss.sum()

## Class distribution

In [ ]:
iea = pd.read_excel("../data/iea.xlsx")
noniea = pd.read_excel("../data/noniea.xlsx")
print("IEA:", iea.shape)
print("NONIEA:", noniea.shape)


IEA: (11010, 7)
NONIEA: (3687, 7)


In [31]:
iea['class'].value_counts()

class
2    1566
3    1248
1     242
6     230
5     204
4     183
7      14
Name: count, dtype: int64

In [32]:
iea['subclass'].value_counts()

subclass
1     990
3     703
7     571
6     363
4     346
2     325
5     205
8      50
11     33
14     26
9      24
15     14
17     13
16      8
10      8
18      3
99      3
13      1
12      1
Name: count, dtype: int64

In [33]:
iea['subsubclass'].value_counts()

subsubclass
1     1594
2      533
4      363
3      351
7      154
19     105
11      88
13      87
99      75
26      65
10      54
21      51
12      41
25      34
8       26
5       24
16      13
14      11
6        7
18       5
22       5
98       1
Name: count, dtype: int64

In [34]:
iea['substrate'].value_counts()

substrate
27     357
1      335
12     263
3      198
2      161
      ... 
86       2
47       1
80       1
170      1
94       1
Name: count, Length: 105, dtype: int64

## NONIEA

In [35]:
iea['class'].value_counts()

class
2    1566
3    1248
1     242
6     230
5     204
4     183
7      14
Name: count, dtype: int64

In [36]:
iea['subclass'].value_counts()

subclass
1     990
3     703
7     571
6     363
4     346
2     325
5     205
8      50
11     33
14     26
9      24
15     14
17     13
16      8
10      8
18      3
99      3
13      1
12      1
Name: count, dtype: int64

In [37]:
iea['subsubclass'].value_counts()

subsubclass
1     1594
2      533
4      363
3      351
7      154
19     105
11      88
13      87
99      75
26      65
10      54
21      51
12      41
25      34
8       26
5       24
16      13
14      11
6        7
18       5
22       5
98       1
Name: count, dtype: int64

In [38]:
iea['substrate'].value_counts()

substrate
27     357
1      335
12     263
3      198
2      161
      ... 
86       2
47       1
80       1
170      1
94       1
Name: count, Length: 105, dtype: int64

In [16]:
class BERTMultiLabelModel(torch.nn.Module):
    def __init__(self):
        """
        In the constructor we instantiate four parameters and assign them as
        member parameters.
        """
        super().__init__()
        self.biobert = AutoModel.from_pretrained("dmis-lab/biobert-v1.1")
        self.class_linear = nn.Linear(768, 7)
        self.subclass_linear = nn.Linear(768, 99)
        self.subsubclass_linear = nn.Linear(768, 99)
        self.substrate_linear = nn.Linear(768, 297)
        self.softmax = nn.Softmax(dim = 1)

    def forward(self, x):
        """
        In the forward function we accept a Tensor of input data and we must return
        a Tensor of output data. We can use Modules defined in the constructor as
        well as arbitrary operators on Tensors.
        """
        
        self.embeddings = self.biobert(x['input_ids'].cuda(), attention_mask = x['attention_mask'].cuda())[1]
        self.output_class = self.softmax(self.class_linear(self.embeddings))
        self.output_subclass = self.softmax(self.subclass_linear(self.embeddings))
        self.output_subsubclass = self.softmax(self.subsubclass_linear(self.embeddings))
        self.output_substrate = self.softmax(self.substrate_linear(self.embeddings))
        self.final_output = dict()
        self.final_output["output_class"] = self.output_class
        self.final_output["output_subclass"] = self.output_subclass
        self.final_output["output_subsubclass"] = self.output_subsubclass
        self.final_output["output_substrate"] = self.output_substrate
        
        return self.final_output

In [162]:
class BERTMultiLabelModel1(torch.nn.Module):
    def __init__(self):
        """
        In the constructor we instantiate four parameters and assign them as
        member parameters.
        """
        super().__init__()
        self.biobert = AutoModel.from_pretrained("dmis-lab/biobert-v1.1")
        self.class_linear = nn.Linear(768, 7)
        #self.softmax = nn.Softmax(dim = 1)

    def forward(self, x):
        """
        In the forward function we accept a Tensor of input data and we must return
        a Tensor of output data. We can use Modules defined in the constructor as
        well as arbitrary operators on Tensors.
        """
        
        self.embeddings = self.biobert(x['input_ids'].cuda(), attention_mask = x['attention_mask'].cuda())[1]
        self.output_class = self.class_linear(self.embeddings)
        self.final_output = dict()
        self.final_output["output_class"] = self.output_class
        
        return self.final_output

In [163]:
class BERTMultiLabelModel2(torch.nn.Module):
    def __init__(self):
        """
        In the constructor we instantiate four parameters and assign them as
        member parameters.
        """
        super().__init__()
        self.biobert = AutoModel.from_pretrained("dmis-lab/biobert-v1.1")
        self.subclass_linear = nn.Linear(768, 99)
        #self.softmax = nn.Softmax(dim = 1)

    def forward(self, x):
        """
        In the forward function we accept a Tensor of input data and we must return
        a Tensor of output data. We can use Modules defined in the constructor as
        well as arbitrary operators on Tensors.
        """
        
        self.embeddings = self.biobert(x['input_ids'].cuda(), attention_mask = x['attention_mask'].cuda())[1]
        self.output_subclass = self.subclass_linear(self.embeddings)
        self.final_output = dict()
        self.final_output["output_subclass"] = self.output_subclass
        
        return self.final_output

In [164]:
class BERTMultiLabelModel3(torch.nn.Module):
    def __init__(self):
        """
        In the constructor we instantiate four parameters and assign them as
        member parameters.
        """
        super().__init__()
        self.biobert = AutoModel.from_pretrained("dmis-lab/biobert-v1.1")
        self.substrate_linear = nn.Linear(768, 297)
        #self.softmax = nn.Softmax(dim = 1)

    def forward(self, x):
        """
        In the forward function we accept a Tensor of input data and we must return
        a Tensor of output data. We can use Modules defined in the constructor as
        well as arbitrary operators on Tensors.
        """
        
        self.embeddings = self.biobert(x['input_ids'].cuda(), attention_mask = x['attention_mask'].cuda())[1]
        self.output_subclass = self.substrate_linear(self.embeddings)
        self.final_output = dict()
        self.final_output["output_subclass"] = self.output_subclass
        
        return self.final_output

## IEA Size

In [165]:
model = BERTMultiLabelModel()
model_class = BERTMultiLabelModel1()
model_subclass = BERTMultiLabelModel2()
model_subsubclass = BERTMultiLabelModel2()
model_substrate = BERTMultiLabelModel3()

### Shared Embeddings

In [153]:
param_size = 0
for param in model.parameters():
    param_size += param.nelement() * param.element_size()
buffer_size = 0
for buffer in model.buffers():
    buffer_size += buffer.nelement() * buffer.element_size()

size_all_mb = (param_size + buffer_size) / 1024**2
print('model size: {:.3f}MB'.format(size_all_mb))

model size: 414.651MB


In [168]:
print("shared embedding: ", sum(p.numel() for p in model.parameters()))
print("shared class embedding: ", sum(p.numel() for p in model_class.parameters()))
print("shared subclass embedding: ", sum(p.numel() for p in model_subclass.parameters()))
print("shared subsubclass embedding: ", sum(p.numel() for p in model_subsubclass.parameters()))
print("shared serial number embedding: ", sum(p.numel() for p in model_substrate.parameters()))

shared embedding:  108696310
shared class embedding:  108315655
shared subclass embedding:  108386403
shared subsubclass embedding:  108386403
shared serial number embedding:  108538665


In [160]:
param_size = 0

for param in model.parameters():
    param_size += param.nelement()

param_size

108696310

In [156]:
for param in model.parameters():
    print(param.element_size())

4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4
4


### Non Shared Embeddings

In [46]:
param_size = 0
for param in model_class.parameters():
    param_size += param.nelement() * param.element_size()
buffer_size = 0
for buffer in model_class.buffers():
    buffer_size += buffer.nelement() * buffer.element_size()

size_all_mb = (param_size + buffer_size) / 1024**2
print('model size: {:.3f}MB'.format(size_all_mb))

model size: 413.199MB


In [47]:
param_size = 0
for param in model_subclass.parameters():
    param_size += param.nelement() * param.element_size()
buffer_size = 0
for buffer in model_subclass.buffers():
    buffer_size += buffer.nelement() * buffer.element_size()

size_all_mb = (param_size + buffer_size) / 1024**2
print('model size: {:.3f}MB'.format(size_all_mb))

model size: 413.469MB


In [48]:
param_size = 0
for param in model_subsubclass.parameters():
    param_size += param.nelement() * param.element_size()
buffer_size = 0
for buffer in model_subsubclass.buffers():
    buffer_size += buffer.nelement() * buffer.element_size()

size_all_mb = (param_size + buffer_size) / 1024**2
print('model size: {:.3f}MB'.format(size_all_mb))

model size: 413.469MB


In [49]:
param_size = 0
for param in model_substrate.parameters():
    param_size += param.nelement() * param.element_size()
buffer_size = 0
for buffer in model_substrate.buffers():
    buffer_size += buffer.nelement() * buffer.element_size()

size_all_mb = (param_size + buffer_size) / 1024**2
print('model size: {:.3f}MB'.format(size_all_mb))

model size: 414.050MB


## NONIEA Size

In [50]:
model = BERTMultiLabelModel()
model_class = BERTMultiLabelModel1()
model_subclass = BERTMultiLabelModel2()
model_subsubclass = BERTMultiLabelModel2()
model_substrate = BERTMultiLabelModel3()

### Shared Embeddings

In [51]:
param_size = 0
for param in model.parameters():
    param_size += param.nelement() * param.element_size()
buffer_size = 0
for buffer in model.buffers():
    buffer_size += buffer.nelement() * buffer.element_size()

size_all_mb = (param_size + buffer_size) / 1024**2
print('model size: {:.3f}MB'.format(size_all_mb))

model size: 414.651MB


### Non Shared Embeddings

In [52]:
param_size = 0
for param in model_class.parameters():
    param_size += param.nelement() * param.element_size()
buffer_size = 0
for buffer in model_class.buffers():
    buffer_size += buffer.nelement() * buffer.element_size()

size_all_mb = (param_size + buffer_size) / 1024**2
print('model size: {:.3f}MB'.format(size_all_mb))

model size: 413.199MB


In [53]:
param_size = 0
for param in model_subclass.parameters():
    param_size += param.nelement() * param.element_size()
buffer_size = 0
for buffer in model_subclass.buffers():
    buffer_size += buffer.nelement() * buffer.element_size()

size_all_mb = (param_size + buffer_size) / 1024**2
print('model size: {:.3f}MB'.format(size_all_mb))

model size: 413.469MB


In [54]:
param_size = 0
for param in model_subsubclass.parameters():
    param_size += param.nelement() * param.element_size()
buffer_size = 0
for buffer in model_subsubclass.buffers():
    buffer_size += buffer.nelement() * buffer.element_size()

size_all_mb = (param_size + buffer_size) / 1024**2
print('model size: {:.3f}MB'.format(size_all_mb))

model size: 413.469MB


In [55]:
param_size = 0
for param in model_substrate.parameters():
    param_size += param.nelement() * param.element_size()
buffer_size = 0
for buffer in model_substrate.buffers():
    buffer_size += buffer.nelement() * buffer.element_size()

size_all_mb = (param_size + buffer_size) / 1024**2
print('model size: {:.3f}MB'.format(size_all_mb))

model size: 414.050MB


## IEA

## IEA focal-loss training

In [19]:
def freeze_layer(layer):
    for name, param in model.named_parameters():
        if name == '{}.weight'.format(layer) or name == '{}.bias'.format(layer):
            param.requires_grad = False
            print(f"Parameter {name} is frozen.")

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
#optimizer = ADOPT(model.parameters(), lr=0.5, decoupled = True)
epochs = 20
loss_function = nn.CrossEntropyLoss()

f1_class = []
f1_subclass = []
f1_subsubclass = []
f1_substrate = []

patienceClass = int(0.1 * epochs)
patienceSubclass = int(0.3 * epochs)
patienceSubsubclass = int(0.3 * epochs)
patienceSubstrate = int(0.4 * epochs)

patience_class = 0
patience_subclass = 0
patience_subsubclass = 0
patience_substrate = 0

class_frozen = False
subclass_frozen = False
subsubclass_frozen = False
substrate_frozen = False

model.cuda()

for epoch in range(epochs):
    model.train()
    train_pbar = tqdm(train_loader, leave=True, total=len(train_loader))
    list_hyp_class = []
    list_hyp_class_label = []
    list_hyp_subclass = []
    list_hyp_subclass_label = []
    list_hyp_subsubclass = []
    list_hyp_subsubclass_label = []
    list_hyp_substrate = []
    list_hyp_substrate_label = []
    
    for batch in train_pbar:
        optimizer.zero_grad()
        
        class_onehot = torch.zeros((batch['labels'].size(0), 7))
        class_onehot[torch.arange(batch['labels'].size(0)), batch['labels'][:, 0] - 1] = 1
        subclass_onehot = torch.zeros((batch['labels'].size(0), 99))
        subclass_onehot[torch.arange(batch['labels'].size(0)), batch['labels'][:, 1] - 1] = 1
        subsubclass_onehot = torch.zeros((batch['labels'].size(0), 99))
        subsubclass_onehot[torch.arange(batch['labels'].size(0)), batch['labels'][:, 2] - 1] = 1
        substrate_onehot = torch.zeros((batch['labels'].size(0), 297))
        substrate_onehot[torch.arange(batch['labels'].size(0)), batch['labels'][:, 3] - 1] = 1
        
        outputs = model(batch)

        loss_class = focal_loss(outputs['output_class'], class_onehot.cuda(), 0.25, 2)
        loss_subclass = focal_loss(outputs['output_subclass'], subclass_onehot.cuda(), 0.25, 2)
        loss_subsubclass = focal_loss(outputs['output_subsubclass'], subsubclass_onehot.cuda(), 0.25, 2)
        loss_substrate = focal_loss(outputs['output_substrate'], substrate_onehot.cuda(), 0.25, 2)
        loss = loss_class + loss_subclass + loss_subsubclass + loss_substrate
        loss.backward()

        optimizer.step()

        hyp_class = outputs['output_class'].argmax(dim = 1)
        list_hyp_class.extend(hyp_class.cpu().numpy() + 1)
        list_hyp_class_label.extend(batch['labels'][:, 0].cpu().numpy())
        hyp_subclass = outputs['output_subclass'].argmax(dim = 1)
        list_hyp_subclass.extend(hyp_subclass.cpu().numpy() + 1)
        list_hyp_subclass_label.extend(batch['labels'][:, 1].cpu().numpy())
        hyp_subsubclass = outputs['output_subsubclass'].argmax(dim = 1)
        list_hyp_subsubclass.extend(hyp_subsubclass.cpu().numpy() + 1)
        list_hyp_subsubclass_label.extend(batch['labels'][:, 2].cpu().numpy())
        hyp_substrate = outputs['output_substrate'].argmax(dim = 1)
        list_hyp_substrate.extend(hyp_substrate.cpu().numpy() + 1)
        list_hyp_substrate_label.extend(batch['labels'][:, 3].cpu().numpy())

        class_metrics = document_metrics(list_hyp_class, list_hyp_class_label)
        subclass_metrics = document_metrics(list_hyp_subclass, list_hyp_subclass_label)
        subsubclass_metrics = document_metrics(list_hyp_subsubclass, list_hyp_subsubclass_label)
        substrate_metrics = document_metrics(list_hyp_substrate, list_hyp_substrate_label)

    #torch.save(model.state_dict(), "bert_uninformed_iea_{}.pth".format(epoch + 1))
    #torch.save(optimizer.state_dict(), "optimizer_bert_uninformed_iea_{}.pth".format(epoch + 1))
    #\nLOSS CLASS: {}\nLOSS SUBCLASS: {}\nLOSS SUBSUBCLASS: {}\n
    print("\n(Epoch {})\nLOSS: {}\nLOSS CLASS: {}\nLOSS SUBCLASS: {}\nLOSS SUBSUBCLASS: {}\nLOSS SUBSTRATE: {}\nCLASS {}\nSUBCLASS {}\nSUBSUBCLASS {}\nSUBSTRATE {}".format(epoch + 1, 
                                                                                                                                                                                  loss, 
                                                                                                                                                                                  loss_class, 
                                                                                                                                                                                  loss_subclass, 
                                                                                                                                                                                  loss_subsubclass, 
                                                                                                                                                                                  loss_substrate, 
                                                                                                                                                                                  metrics_to_string(class_metrics), 
                                                                                                                                                                                  metrics_to_string(subclass_metrics), 
                                                                                                                                                                                  metrics_to_string(subsubclass_metrics), 
                                                                                                                                                                                  metrics_to_string(substrate_metrics)))
    
    model.eval()

    with torch.no_grad():
        list_hyp_class = []
        list_hyp_class_label = []
        list_hyp_subclass = []
        list_hyp_subclass_label = []
        list_hyp_subsubclass = []
        list_hyp_subsubclass_label = []
        list_hyp_substrate = []
        list_hyp_substrate_label = []
        
        for batch in tqdm(test_loader):
            output = model(batch)
    
            hyp_class = output['output_class'].argmax(dim = 1)
            list_hyp_class.extend(hyp_class.cpu().numpy() + 1)
            list_hyp_class_label.extend(batch['labels'][:, 0].cpu().numpy())
            hyp_subclass = output['output_subclass'].argmax(dim = 1)
            list_hyp_subclass.extend(hyp_subclass.cpu().numpy() + 1)
            list_hyp_subclass_label.extend(batch['labels'][:, 1].cpu().numpy())
            hyp_subsubclass = output['output_subsubclass'].argmax(dim = 1)
            list_hyp_subsubclass.extend(hyp_subsubclass.cpu().numpy() + 1)
            list_hyp_subsubclass_label.extend(batch['labels'][:, 2].cpu().numpy())
            hyp_substrate = output['output_substrate'].argmax(dim = 1)
            list_hyp_substrate.extend(hyp_substrate.cpu().numpy() + 1)
            list_hyp_substrate_label.extend(batch['labels'][:, 3].cpu().numpy())
    
            class_metrics = document_metrics(list_hyp_class, list_hyp_class_label)
            subclass_metrics = document_metrics(list_hyp_subclass, list_hyp_subclass_label)
            subsubclass_metrics = document_metrics(list_hyp_subsubclass, list_hyp_subsubclass_label)
            substrate_metrics = document_metrics(list_hyp_substrate, list_hyp_substrate_label)
    
        print("\n(Epoch {})\nCLASS {}\nSUBCLASS {}\nSUBSUBCLASS {}\nSUBSTRATE {}".format(epoch + 1, metrics_to_string(class_metrics), 
                                                                                                         metrics_to_string(subclass_metrics), 
                                                                                                         metrics_to_string(subsubclass_metrics), 
                                                                                                         metrics_to_string(substrate_metrics)))
        '''
        if len(f1_class) < 3:
            f1_class.append(f1_score(list_hyp_class, list_hyp_class_label, average = 'macro'))
        else:
#            f1_class_copy = f1_class[-3:].copy()
#            f1_class_copy.sort()
            if f1_score(list_hyp_class, list_hyp_class_label, average = 'macro') - f1_class[-1] < 0.1:
                patience_class += 1
            if patience_class >= patienceClass:
                freeze_layer("class_linear")
                class_frozen = True
            f1_class.append(f1_score(list_hyp_class, list_hyp_class_label, average = 'macro'))
        print('patience class: ', patience_class)
        if len(f1_subclass) < 3:
            f1_subclass.append(f1_score(list_hyp_subclass, list_hyp_subclass_label, average = 'macro'))
        else:
#            f1_subclass_copy = f1_subclass[-3:].copy()
#            f1_subclass_copy.sort()
            if f1_score(list_hyp_subclass, list_hyp_subclass_label, average = 'macro') - f1_subclass[-1] < 0.05:
                patience_subclass += 1
            if patience_subclass >= patienceSubclass:
                freeze_layer("subclass_linear")
                subclass_frozen = True
            f1_subclass.append(f1_score(list_hyp_subclass, list_hyp_subclass_label, average = 'macro'))
        print('patience subclass: ', patience_subclass)
        if len(f1_subsubclass) < 3:
            f1_subsubclass.append(f1_score(list_hyp_subsubclass, list_hyp_subsubclass_label, average = 'macro'))
        else:
#            f1_subsubclass_copy = f1_subsubclass[-3:].copy()
#            f1_subsubclass_copy.sort()
            if f1_score(list_hyp_subsubclass, list_hyp_subsubclass_label, average = 'macro') - f1_subsubclass[-1] < 0.05:
                patience_subsubclass += 1
            if patience_subsubclass >= patienceSubsubclass:
                freeze_layer("subsubclass_linear")
                subsubclass_frozen = True
            f1_subsubclass.append(f1_score(list_hyp_subsubclass, list_hyp_subsubclass_label, average = 'macro'))
        print('patience subsubclass: ', patience_subsubclass)
        if len(f1_substrate) < 3:
            f1_substrate.append(f1_score(list_hyp_substrate, list_hyp_substrate_label, average = 'macro'))
        else:
#            f1_substrate_copy = f1_substrate[-3:].copy()
#            f1_substrate_copy.sort()
            if f1_score(list_hyp_substrate, list_hyp_substrate_label, average = 'macro') - f1_substrate[-1] < 0.01:
                patience_substrate += 1
            if patience_substrate >= patienceSubstrate:
                freeze_layer("substrate_linear")
                substrate_frozen = True
            f1_substrate.append(f1_score(list_hyp_substrate, list_hyp_substrate_label, average = 'macro'))
        print('patience substrate: ', patience_substrate)
    if class_frozen and subclass_frozen and subsubclass_frozen and substrate_frozen:
        print('stopped at epoch {}'.format(epoch + 1))
        break'''

100%|██████████| 482/482 [07:37<00:00,  1.05it/s]



(Epoch 1)
LOSS: 9.992069244384766
LOSS CLASS: 0.24669258296489716
LOSS SUBCLASS: 1.0729942321777344
LOSS SUBSUBCLASS: 1.4833815097808838
LOSS SUBSTRATE: 7.189001083374023
CLASS Accuracy:0.834 F1 Score:0.757 Precision:0.687 Recall:0.903
SUBCLASS Accuracy:0.671 F1 Score:0.236 Precision:0.217 Recall:0.304
SUBSUBCLASS Accuracy:0.674 F1 Score:0.190 Precision:0.164 Recall:0.326
SUBSTRATE Accuracy:0.182 F1 Score:0.034 Precision:0.037 Recall:0.062


100%|██████████| 207/207 [01:05<00:00,  3.17it/s]



(Epoch 1)
CLASS Accuracy:0.959 F1 Score:0.953 Precision:0.940 Recall:0.969
SUBCLASS Accuracy:0.883 F1 Score:0.378 Precision:0.377 Recall:0.386
SUBSUBCLASS Accuracy:0.821 F1 Score:0.312 Precision:0.332 Recall:0.310
SUBSTRATE Accuracy:0.336 F1 Score:0.092 Precision:0.110 Recall:0.105


100%|██████████| 482/482 [07:39<00:00,  1.05it/s]


...


(Epoch 19)
CLASS Accuracy:0.983 F1 Score:0.977 Precision:0.974 Recall:0.981
SUBCLASS Accuracy:0.981 F1 Score:0.950 Precision:0.942 Recall:0.963
SUBSUBCLASS Accuracy:0.977 F1 Score:0.931 Precision:0.928 Recall:0.944
SUBSTRATE Accuracy:0.953 F1 Score:0.957 Precision:0.958 Recall:0.964


100%|██████████| 482/482 [07:36<00:00,  1.06it/s]



(Epoch 20)
LOSS: 0.016210278496146202
LOSS CLASS: 0.00025187525898218155
LOSS SUBCLASS: 0.001994140213355422
LOSS SUBSUBCLASS: 0.0030128315556794405
LOSS SUBSTRATE: 0.010951431468129158
CLASS Accuracy:0.992 F1 Score:0.990 Precision:0.990 Recall:0.989
SUBCLASS Accuracy:0.988 F1 Score:0.933 Precision:0.909 Recall:0.966
SUBSUBCLASS Accuracy:0.987 F1 Score:0.962 Precision:0.955 Recall:0.970
SUBSTRATE Accuracy:0.966 F1 Score:0.955 Precision:0.948 Recall:0.965


100%|██████████| 207/207 [01:04<00:00,  3.23it/s]


(Epoch 20)
CLASS Accuracy:0.984 F1 Score:0.979 Precision:0.977 Recall:0.981
SUBCLASS Accuracy:0.980 F1 Score:0.958 Precision:0.950 Recall:0.971
SUBSUBCLASS Accuracy:0.967 F1 Score:0.920 Precision:0.913 Recall:0.936
SUBSTRATE Accuracy:0.940 F1 Score:0.937 Precision:0.946 Recall:0.945


## IEA evaluation and hierarchical correctness

In [ ]:
model.eval()

list_hyp_class = []
list_hyp_subclass = []
list_hyp_subsubclass = []
list_hyp_substrate = []

prob_class = []
prob_subclass = []
prob_subsubclass = []
prob_substrate = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        output = model(batch)

        class_tensor = output["output_class"]
        subclass_tensor = output["output_subclass"]
        subsubclass_tensor = output["output_subsubclass"]
        substrate_tensor = output["output_substrate"]

        class_pred = class_tensor.argmax(dim=1)
        subclass_pred = subclass_tensor.argmax(dim=1)
        subsubclass_pred = subsubclass_tensor.argmax(dim=1)
        substrate_pred = substrate_tensor.argmax(dim=1)

        list_hyp_class.extend((class_pred.cpu().numpy() + 1).tolist())
        list_hyp_subclass.extend((subclass_pred.cpu().numpy() + 1).tolist())
        list_hyp_subsubclass.extend((subsubclass_pred.cpu().numpy() + 1).tolist())
        list_hyp_substrate.extend((substrate_pred.cpu().numpy() + 1).tolist())

        prob_class.extend(class_tensor.cpu().numpy().tolist())
        prob_subclass.extend(subclass_tensor.cpu().numpy().tolist())
        prob_subsubclass.extend(subsubclass_tensor.cpu().numpy().tolist())
        prob_substrate.extend(substrate_tensor.cpu().numpy().tolist())

y_copy = y_test.reset_index(drop=True).copy()
y_copy["pred_class"] = list_hyp_class
y_copy["pred_subclass"] = list_hyp_subclass
y_copy["pred_subsubclass"] = list_hyp_subsubclass
y_copy["pred_substrate"] = list_hyp_substrate

class_metrics = document_metrics(y_copy["pred_class"], y_copy["class"])
subclass_metrics = document_metrics(y_copy["pred_subclass"], y_copy["subclass"])
subsubclass_metrics = document_metrics(y_copy["pred_subsubclass"], y_copy["subsubclass"])
substrate_metrics = document_metrics(y_copy["pred_substrate"], y_copy["substrate"])

print("CLASS", metrics_to_string(class_metrics))
print("SUBCLASS", metrics_to_string(subclass_metrics))
print("SUBSUBCLASS", metrics_to_string(subsubclass_metrics))
print("SUBSTRATE", metrics_to_string(substrate_metrics))

y_copy.head()


CLASS Accuracy:0.959 F1 Score:0.953 Precision:0.940 Recall:0.969
SUBCLASS Accuracy:0.883 F1 Score:0.378 Precision:0.377 Recall:0.386
SUBSUBCLASS Accuracy:0.821 F1 Score:0.312 Precision:0.332 Recall:0.310
SUBSTRATE Accuracy:0.336 F1 Score:0.092 Precision:0.110 Recall:0.105


In [ ]:
pred_class_prob = []
pred_subclass_prob = []
pred_subsubclass_prob = []
pred_substrate_prob = []

for i, (values_class, values_subclass, values_subsubclass, values_substrate) in enumerate(
    zip(prob_class, prob_subclass, prob_subsubclass, prob_substrate)
):
    pred_class_prob.append(values_class[y_copy.loc[i, 'pred_class'] - 1])
    pred_subclass_prob.append(values_subclass[y_copy.loc[i, 'pred_subclass'] - 1])
    pred_subsubclass_prob.append(values_subsubclass[y_copy.loc[i, 'pred_subsubclass'] - 1])
    pred_substrate_prob.append(values_substrate[y_copy.loc[i, 'pred_substrate'] - 1])

pred_prob = pd.DataFrame({
    'class': pred_class_prob,
    'subclass': pred_subclass_prob,
    'subsubclass': pred_subsubclass_prob,
    'substrate': pred_substrate_prob
})

correct_index = y_copy[
    (y_copy['class'] == y_copy['pred_class']) &
    (y_copy['subclass'] == y_copy['pred_subclass']) &
    (y_copy['subsubclass'] == y_copy['pred_subsubclass']) &
    (y_copy['substrate'] == y_copy['pred_substrate'])
].index

pred_prob.loc[correct_index].head()


,class,subclass,subsubclass,substrate
0,0.982459,0.865844,0.989563,0.968720
1,0.983805,0.847179,0.983396,0.877041
2,0.957093,0.884692,0.970310,0.953905
3,0.977661,0.942521,0.885765,0.968008
4,0.990164,0.962688,0.955779,0.967337
